In [3]:
%load_ext autoreload
%autoreload 2

import holidays
import httpx
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import os

ML_ENGINE_BASE_URL = os.environ.get("ML_ENGINE_BASE_URL", "http://localhost:8001").rstrip("/")
REALIZED_PATH = "/api/v1/energy/consumption/realized"
url = f"{ML_ENGINE_BASE_URL}{REALIZED_PATH}"
params = {
    "start": "2024-01-01",
    "end": "2024-01-10",
}
pd.set_option("display.max_rows", 100)

response = httpx.get(url, params=params, timeout=30.0)
response.raise_for_status()
data = response.json()
print(data)

[{'timestamp': '2024-01-01T00:00:00+01:00', 'megawatts': 54033}, {'timestamp': '2024-01-01T00:15:00+01:00', 'megawatts': 54488}, {'timestamp': '2024-01-01T00:30:00+01:00', 'megawatts': 53501}, {'timestamp': '2024-01-01T00:45:00+01:00', 'megawatts': 52102}, {'timestamp': '2024-01-01T01:00:00+01:00', 'megawatts': 51292}, {'timestamp': '2024-01-01T01:15:00+01:00', 'megawatts': 51822}, {'timestamp': '2024-01-01T01:30:00+01:00', 'megawatts': 51512}, {'timestamp': '2024-01-01T01:45:00+01:00', 'megawatts': 51731}, {'timestamp': '2024-01-01T02:00:00+01:00', 'megawatts': 51333}, {'timestamp': '2024-01-01T02:15:00+01:00', 'megawatts': 51790}, {'timestamp': '2024-01-01T02:30:00+01:00', 'megawatts': 51315}, {'timestamp': '2024-01-01T02:45:00+01:00', 'megawatts': 50741}, {'timestamp': '2024-01-01T03:00:00+01:00', 'megawatts': 50122}, {'timestamp': '2024-01-01T03:15:00+01:00', 'megawatts': 49238}, {'timestamp': '2024-01-01T03:30:00+01:00', 'megawatts': 48399}, {'timestamp': '2024-01-01T03:45:00+01:0

In [5]:
# Transform the data to TimeSeries

# Transform the data to dataframe
df = pd.DataFrame(data)

df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

# set the timestamp as index
df.set_index("timestamp", inplace=True)

df.head()

,megawatts
timestamp,
2023-12-31 23:00:00+00:00,54033
2023-12-31 23:15:00+00:00,54488
2023-12-31 23:30:00+00:00,53501
2023-12-31 23:45:00+00:00,52102
2024-01-01 00:00:00+00:00,51292


In [6]:
# Resampling the data to hourly (mean of the megawatts values)

df_resampled = df.resample("1h").mean()

df_resampled.head()

,megawatts
timestamp,
2023-12-31 23:00:00+00:00,53531.00
2024-01-01 00:00:00+00:00,51589.25
2024-01-01 01:00:00+00:00,51294.75
2024-01-01 02:00:00+00:00,48886.50
2024-01-01 03:00:00+00:00,46812.25


In [7]:
# Feature engineering
# We will create new features (columns) bases on our actual data to improve the model performance

# 1. Day of the week
df_resampled["day_of_week"] = df_resampled.index.dayofweek
df_resampled["hour_of_day"] = df_resampled.index.hour
df_resampled["month"] = df_resampled.index.month

df_resampled["is_weekend"] = df_resampled.index.dayofweek.isin([5, 6]).astype(int)

# bank holidays
years_present = df_resampled.index.year.unique().tolist()
fr_bank_holidays = holidays.France(years=years_present)

# Holiday lookup uses calendar dates; compare via .index.date, not raw timestamps.
df_resampled["is_bank_holiday"] = (
    pd.Index(df_resampled.index.date).isin(fr_bank_holidays).astype(int)
)

df_resampled.head()

,megawatts,day_of_week,hour_of_day,month,is_weekend,is_bank_holiday
timestamp,,,,,,
2023-12-31 23:00:00+00:00,53531.00,6,23,12,1,0
2024-01-01 00:00:00+00:00,51589.25,0,0,1,0,1
2024-01-01 01:00:00+00:00,51294.75,0,1,1,0,1
2024-01-01 02:00:00+00:00,48886.50,0,2,1,0,1
2024-01-01 03:00:00+00:00,46812.25,0,3,1,0,1


In [8]:
# Suppress the empty data
df_resampled.isna().sum()

# Interpolate NaNs linearly (keeps continuity; dropping would leave gaps).
df_resampled = df_resampled.interpolate(method="linear")

# Drop edge NaNs that interpolation cannot fill (series start/end).
df_resampled = df_resampled.dropna()
df_resampled.head()

# Export CSV; index is the timestamp and is written as a column when needed.
df_resampled.to_csv("../data/processed/energy_consumption_realized_resampled.csv", index=True)